In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [2]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
 
FULL_PATH = r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df_full.parquet"

In [3]:
df = pd.read_parquet(FULL_PATH)
print(f"Loaded {FULL_PATH}")
print("Shape:", df.shape)
assert df.shape[0] == 178384, f"Row count mismatch: expected 178384, got {df.shape[0]}"
 
TARGET = "question_recommendation_nps_a"

Loaded C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df_full.parquet
Shape: (178384, 168)


## 3 classes


In [4]:
def nps_category(score):
    if score <= 6:
        return 0  # Detractor
    elif score <= 8:
        return 1  # Passive
    else:
        return 2  # Promoter
 
 
y_cat = df[TARGET].apply(nps_category)
print("\nNPS category distribution:")
print(y_cat.value_counts(normalize=True).sort_index())


NPS category distribution:
question_recommendation_nps_a
0    0.166052
1    0.324188
2    0.509760
Name: proportion, dtype: float64


In [5]:
y_cat = df[TARGET].apply(nps_category)
print("\nNPS category distribution:")
print(y_cat.value_counts(normalize=True).sort_index())
 
print("\nmetadata_price already in df:", "metadata_price" in df.columns)
print(df["metadata_price"].describe())
print("\nstaff_composite already in df:", "staff_composite" in df.columns)
print(df["staff_composite"].describe())


NPS category distribution:
question_recommendation_nps_a
0    0.166052
1    0.324188
2    0.509760
Name: proportion, dtype: float64

metadata_price already in df: True
count    178384.000000
mean         94.021885
std          62.269216
min           3.000000
25%          49.000000
50%          75.000000
75%         119.000000
max         553.000000
Name: metadata_price, dtype: float64

staff_composite already in df: True
count    178384.0
mean     4.115774
std      0.798811
min           1.0
25%           4.0
50%           4.0
75%           5.0
max           5.0
Name: staff_composite, dtype: Float64


In [6]:
print("\n last_clean_score vs clean_score_routine ")
print("corr:", df["last_clean_score"].corr(df["clean_score_routine"]))
print("equal (exact match) fraction:", (df["last_clean_score"] == df["clean_score_routine"]).mean())
 
print("\n hours_since_last_clean vs clean_hours_since_routine ")
print("corr:", df["hours_since_last_clean"].corr(df["clean_hours_since_routine"]))
print("equal (exact match) fraction:", (df["hours_since_last_clean"] == df["clean_hours_since_routine"]).mean())
 
print("\n clean_recency_encoded ")
print(df["clean_recency_encoded"].value_counts(dropna=False).sort_index())
 
print("\n pm_days_since_*_has_data vs is_channel (checking for near-redundancy) ")
for col in ["pm_days_since_interior_has_data", "pm_days_since_catering_has_data",
            "pm_days_since_comms_has_data", "pm_days_since_toilet_has_data",
            "pm_days_since_climate_has_data"]:
    print(f"\n{col}:")
    print(pd.crosstab(df[col], df["is_channel"]))
 
print("\n cause_select_Delayed_NoCauseRecord row count (expect ~11023) ")
print(int(df["cause_select_Delayed_NoCauseRecord"].sum()))


 last_clean_score vs clean_score_routine 
corr: -0.02900688923472987
equal (exact match) fraction: 0.5535642210063683

 hours_since_last_clean vs clean_hours_since_routine 
corr: 0.9057599959729132
equal (exact match) fraction: 0.5535642210063683

 clean_recency_encoded 
clean_recency_encoded
0.0     2906
1.0     4424
1.5    53978
2.0    36446
3.0    38661
4.0    41969
Name: count, dtype: int64

 pm_days_since_*_has_data vs is_channel (checking for near-redundancy) 

pm_days_since_interior_has_data:
is_channel                           0       1
pm_days_since_interior_has_data               
0                                60192       0
1                                 1927  116265

pm_days_since_catering_has_data:
is_channel                           0       1
pm_days_since_catering_has_data               
0                                60192       0
1                                 1927  116265

pm_days_since_comms_has_data:
is_channel                        0      1
pm_days_si

In [7]:
print("\n cause_No_Delay arrival_delay_minute distribution")
print(df.loc[df["cause_No_Delay"] == 1, "arrival_delay_minute"].describe())
print("cause_No_Delay row count before fix:", int(df["cause_No_Delay"].sum()))


 cause_No_Delay arrival_delay_minute distribution
count    68932.000000
mean         2.338014
std         17.596329
min        -31.000000
25%         -2.000000
50%          0.000000
75%          0.000000
max        319.000000
Name: arrival_delay_minute, dtype: float64
cause_No_Delay row count before fix: 68932


In [8]:
relabel_mask = df["cause_select_Delayed_NoCauseRecord"] == 1
df["cause_Delayed_NoCauseRecord"] = relabel_mask.astype(int)
df.loc[relabel_mask, "cause_No_Delay"] = 0

In [9]:

print("cause_No_Delay row count after fix:", int(df["cause_No_Delay"].sum()))
print("cause_Delayed_NoCauseRecord row count:", int(df["cause_Delayed_NoCauseRecord"].sum()))
print("cause_No_Delay arrival_delay_minute distribution:")
print(df.loc[df["cause_No_Delay"] == 1, "arrival_delay_minute"].describe())

cause_No_Delay row count after fix: 57909
cause_Delayed_NoCauseRecord row count: 11023
cause_No_Delay arrival_delay_minute distribution:
count    57909.000000
mean        -1.767204
std          2.605569
min        -31.000000
25%         -3.000000
50%         -1.000000
75%          0.000000
max          0.000000
Name: arrival_delay_minute, dtype: float64


In [10]:
cause_group_dummies = [
    "cause_Eurostar_Operations", "cause_Infrastructure", "cause_Passenger_Related",
    "cause_Rolling_Stock", "cause_Traffic_Regulation", "cause_Weather_Safety",
    "cause_Unknown", "cause_Delayed_NoCauseRecord",
]

In [11]:
AD_BREAKS = [10.5, 19.5, 26.5, 46.5, 91.5, 142.5]
AD_LABELS = ["ad_b1_10.5", "ad_b2_19.5", "ad_b3_26.5", "ad_b4_46.5", "ad_b5_91.5", "ad_b6_142.5"]
for thresh, label in zip(AD_BREAKS, AD_LABELS):
    df[label] = (df["arrival_delay_minute"] > thresh).astype(int)

EJ_BREAKS = [10.5, 13.5, 20.5, 31.5]
EJ_LABELS = ["ej_b1_10.5", "ej_b2_13.5", "ej_b3_20.5", "ej_b4_31.5"]
for thresh, label in zip(EJ_BREAKS, EJ_LABELS):
    df[label] = (df["early_journey_delay_minute"] > thresh).astype(int)

delay_dummies = AD_LABELS + EJ_LABELS

In [12]:

print("arrival_delay > 142.5:", (df["arrival_delay_minute"] > 142.5).sum())
print("arrival_delay > 91.5 & <= 142.5:",
      ((df["arrival_delay_minute"] > 91.5) & (df["arrival_delay_minute"] <= 142.5)).sum())


arrival_delay > 142.5: 1543
arrival_delay > 91.5 & <= 142.5: 2921


In [13]:
df["clean_score_routine_channel"] = np.where(df["is_channel"] == 1, df["clean_score_routine"], 0)

# total_pax: high-occupancy breach only, channel-zeroed 
df["total_pax_breach_400"] = np.where(
    (df["is_channel"] == 1) & (df["total_pax"] > 400), 1, 0
)
print("\ntotal_pax_breach_400 row count:", int(df["total_pax_breach_400"].sum()))



total_pax_breach_400 row count: 13901


In [14]:
other_operational_fixed = [
    "clean_score_routine_channel",
    "clean_hours_since_routine",
    "hours_since_last_clean",
    "clean_score_deep",
    "clean_days_since_deep",
    "equip_RUB", "equip_E320",
    "has_equipment_change",
    "has_connection",
    "arrived_early",
    "total_pax_breach_400",
    "is_channel",
]

In [15]:
RESID_BLOCK = [
    "average_days_since_last_exams_resid", "average_days_since_last_exams_has_data",
    "pm_days_since_interior_resid", "pm_days_since_interior_has_data",
    "pm_days_since_catering_resid", "pm_days_since_catering_has_data",
    "pm_days_since_comms_resid",    "pm_days_since_comms_has_data",
    "pm_days_since_toilet_resid",   "pm_days_since_toilet_has_data",
    "pm_days_since_climate_resid",  "pm_days_since_climate_has_data",
    "cm_open_interior_resid", "cm_open_catering_resid", "cm_open_cleaning_resid",
    "cm_open_wifi_resid", "cm_open_climate_resid", "cm_open_toilet_resid", "cm_open_comms_resid",
]
RAW_BLOCK = [
    "average_days_since_last_exams",
    "pm_days_since_interior",
    "pm_days_since_catering",
    "pm_days_since_comms",
    "pm_days_since_toilet",
    "pm_days_since_climate",
    "cm_open_interior", "cm_open_catering", "cm_open_cleaning",
    "cm_open_wifi", "cm_open_climate", "cm_open_toilet", "cm_open_comms",
]

In [16]:
RAW_TO_RESID = {
    "average_days_since_last_exams": ["average_days_since_last_exams_resid", "average_days_since_last_exams_has_data"],
    "pm_days_since_interior": ["pm_days_since_interior_resid", "pm_days_since_interior_has_data"],
    "pm_days_since_catering": ["pm_days_since_catering_resid", "pm_days_since_catering_has_data"],
    "pm_days_since_comms":    ["pm_days_since_comms_resid",   "pm_days_since_comms_has_data"],
    "pm_days_since_toilet":   ["pm_days_since_toilet_resid",  "pm_days_since_toilet_has_data"],
    "pm_days_since_climate":  ["pm_days_since_climate_resid", "pm_days_since_climate_has_data"],
    "cm_open_interior": ["cm_open_interior_resid"],
    "cm_open_catering": ["cm_open_catering_resid"],
    "cm_open_cleaning": ["cm_open_cleaning_resid"],
    "cm_open_wifi": ["cm_open_wifi_resid"],
    "cm_open_climate": ["cm_open_climate_resid"],
    "cm_open_toilet": ["cm_open_toilet_resid"],
    "cm_open_comms": ["cm_open_comms_resid"],
}

In [17]:
core_vars_main = delay_dummies + cause_group_dummies + other_operational_fixed + RESID_BLOCK + ["metadata_price"]
core_vars_raw = delay_dummies + cause_group_dummies + other_operational_fixed + RAW_BLOCK + ["metadata_price"]
print(f"\nMAIN spec (resid+has_data): {len(core_vars_main)} vars")
print(core_vars_main)
print(f"\nALT spec (raw): {len(core_vars_raw)} vars")
print(core_vars_raw)


MAIN spec (resid+has_data): 50 vars
['ad_b1_10.5', 'ad_b2_19.5', 'ad_b3_26.5', 'ad_b4_46.5', 'ad_b5_91.5', 'ad_b6_142.5', 'ej_b1_10.5', 'ej_b2_13.5', 'ej_b3_20.5', 'ej_b4_31.5', 'cause_Eurostar_Operations', 'cause_Infrastructure', 'cause_Passenger_Related', 'cause_Rolling_Stock', 'cause_Traffic_Regulation', 'cause_Weather_Safety', 'cause_Unknown', 'cause_Delayed_NoCauseRecord', 'clean_score_routine_channel', 'clean_hours_since_routine', 'hours_since_last_clean', 'clean_score_deep', 'clean_days_since_deep', 'equip_RUB', 'equip_E320', 'has_equipment_change', 'has_connection', 'arrived_early', 'total_pax_breach_400', 'is_channel', 'average_days_since_last_exams_resid', 'average_days_since_last_exams_has_data', 'pm_days_since_interior_resid', 'pm_days_since_interior_has_data', 'pm_days_since_catering_resid', 'pm_days_since_catering_has_data', 'pm_days_since_comms_resid', 'pm_days_since_comms_has_data', 'pm_days_since_toilet_resid', 'pm_days_since_toilet_has_data', 'pm_days_since_climate_r

In [18]:
X_main = df[core_vars_main].astype(float)
model_main = OrderedModel(y_cat, X_main, distr="logit")
result_main = model_main.fit(method="bfgs", maxiter=500, disp=True)
print(result_main.summary())
print("LL:", result_main.llf, "AIC:", result_main.aic)


Optimization terminated successfully.
         Current function value: 0.955659
         Iterations: 183
         Function evaluations: 194
         Gradient evaluations: 194


C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7047e+05
Model:                              OrderedModel   AIC:                         3.411e+05
Method:                       Maximum Likelihood   BIC:                         3.416e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   17:30:22                                         
No. Observations:                         178384                                         
Df Residuals:                             178332                                         
Df Model:                                     50                                         
                                             coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [19]:
core_vars_main_staff = core_vars_main + ["staff_composite"]
X_main_staff = df[core_vars_main_staff].astype(float)
model_main_staff = OrderedModel(y_cat, X_main_staff, distr="logit")
result_main_staff = model_main_staff.fit(method="bfgs", maxiter=500, disp=True)
print(result_main_staff.summary())
print("LL:", result_main_staff.llf, "AIC:", result_main_staff.aic)
 
print("\n total effects (no staff) vs direct effects (net of staff) ===")
shrink_df = pd.DataFrame({
    "total_effect_coef": result_main.params[core_vars_main],
    "direct_effect_coef": result_main_staff.params[core_vars_main],
})
shrink_df["shrinkage"] = shrink_df["total_effect_coef"] - shrink_df["direct_effect_coef"]
print(shrink_df.round(4).to_string())

Optimization terminated successfully.
         Current function value: 0.889824
         Iterations: 187
         Function evaluations: 196
         Gradient evaluations: 196
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.5873e+05
Model:                              OrderedModel   AIC:                         3.176e+05
Method:                       Maximum Likelihood   BIC:                         3.181e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   18:07:41                                         
No. Observations:                         178384                                         
Df Residuals:                             178331                                         
Df Model:                                     51                                         
               

C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [20]:
X_raw = df[core_vars_raw].astype(float)
model_raw = OrderedModel(y_cat, X_raw, distr="logit")
result_raw = model_raw.fit(method="bfgs", maxiter=500, disp=True)
print(result_raw.summary())
print("LL:", result_raw.llf, "AIC:", result_raw.aic)
 
print("\n RESID+flag vs RAW: model fit ")
print(f"{'spec':25s} {'n_vars':>8s} {'LL':>14s} {'AIC':>12s}")
print(f"{'resid+has_data':25s} {len(core_vars_main):8d} {result_main.llf:14.2f} {result_main.aic:12.2f}")
print(f"{'raw':25s} {len(core_vars_raw):8d} {result_raw.llf:14.2f} {result_raw.aic:12.2f}")
 

Optimization terminated successfully.
         Current function value: 0.955668
         Iterations: 207
         Function evaluations: 217
         Gradient evaluations: 217
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7048e+05
Model:                              OrderedModel   AIC:                         3.410e+05
Method:                       Maximum Likelihood   BIC:                         3.415e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   18:41:40                                         
No. Observations:                         178384                                         
Df Residuals:                             178338                                         
Df Model:                                     44                                         
               

In [21]:
print("\n RESID+flag vs RAW: coefficient comparison by family ")
comp_rows = []
for raw_v, resid_vs in RAW_TO_RESID.items():
    comp_rows.append({
        "family": raw_v,
        "side": "raw",
        "variable": raw_v,
        "coef": result_raw.params[raw_v],
        "p_value": result_raw.pvalues[raw_v],
    })
    for resid_v in resid_vs:
        comp_rows.append({
            "family": raw_v,
            "side": "resid/flag",
            "variable": resid_v,
            "coef": result_main.params[resid_v],
            "p_value": result_main.pvalues[resid_v],
        })
comp_df = pd.DataFrame(comp_rows)
print(comp_df.round(5).to_string(index=False))


 RESID+flag vs RAW: coefficient comparison by family 
                       family       side                               variable     coef  p_value
average_days_since_last_exams        raw          average_days_since_last_exams  0.00001  0.81452
average_days_since_last_exams resid/flag    average_days_since_last_exams_resid -0.00013      NaN
average_days_since_last_exams resid/flag average_days_since_last_exams_has_data -0.00043      NaN
       pm_days_since_interior        raw                 pm_days_since_interior  0.00010  0.06624
       pm_days_since_interior resid/flag           pm_days_since_interior_resid  0.00013      NaN
       pm_days_since_interior resid/flag        pm_days_since_interior_has_data -0.00043      NaN
       pm_days_since_catering        raw                 pm_days_since_catering  0.00051  0.00000
       pm_days_since_catering resid/flag           pm_days_since_catering_resid  0.00053      NaN
       pm_days_since_catering resid/flag        pm_days_since_c

In [22]:
X_vif_const = add_constant(X_main)
vif_data = pd.DataFrame()
vif_data["variable"] = X_vif_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif_const.values, i) for i in range(X_vif_const.shape[1])]
print("\n=== VIF (MAIN spec) ===")
print(vif_data.sort_values("VIF", ascending=False).to_string(index=False))


C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



=== VIF (MAIN spec) ===
                              variable         VIF
        pm_days_since_climate_has_data         inf
       pm_days_since_interior_has_data         inf
average_days_since_last_exams_has_data         inf
       pm_days_since_catering_has_data         inf
   average_days_since_last_exams_resid 6580.327988
          pm_days_since_comms_has_data 3271.921682
            pm_days_since_toilet_resid 2414.963257
           pm_days_since_climate_resid  679.233556
             pm_days_since_comms_resid  179.923945
         pm_days_since_toilet_has_data   78.114870
                            is_channel   55.694720
          pm_days_since_interior_resid   35.484481
           clean_score_routine_channel   21.419930
                                 const   16.859616
             clean_hours_since_routine   12.841543
                hours_since_last_clean   10.802857
                            ej_b2_13.5    9.640065
                            ej_b1_10.5    7.484223
      

In [23]:
cause_select_dummies = [
    "cause_select_A4_LostPathway", "cause_select_I1_Animals", "cause_select_I2_People",
    "cause_select_I3_InfraWorks", "cause_select_I4_InfraFault", "cause_select_I5_TrainsIncident",
    "cause_select_I6_TrafficReg", "cause_select_K1_Catering", "cause_select_M1_Reliability",
    "cause_select_M2_DepotRelease", "cause_select_M3_TrainsetShortage",
    "cause_select_O1_Driving", "cause_select_O2_TrainManager", "cause_select_O3_Planning",
    "cause_select_O4_OCC", "cause_select_Other_rare_within_expanded", "cause_select_S1_Passengers",
    "cause_select_S2_Congestion", "cause_select_S4_TerminalStaff", "cause_select_Unknown",
    "cause_select_Weather_Safety", "cause_select_Delayed_NoCauseRecord",
]

In [24]:
core_vars_select = delay_dummies + cause_select_dummies + other_operational_fixed + RESID_BLOCK + ["metadata_price"]
X_select = df[core_vars_select].astype(float)
model_select = OrderedModel(y_cat, X_select, distr="logit")
result_select = model_select.fit(method="bfgs", maxiter=500, disp=True)
print(result_select.summary())
 

Optimization terminated successfully.
         Current function value: 0.955508
         Iterations: 202
         Function evaluations: 213
         Gradient evaluations: 213


C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7045e+05
Model:                              OrderedModel   AIC:                         3.410e+05
Method:                       Maximum Likelihood   BIC:                         3.417e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   19:36:13                                         
No. Observations:                         178384                                         
Df Residuals:                             178318                                         
Df Model:                                     64                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

In [25]:
select_to_group = {
    "cause_select_M1_Reliability": "Rolling_Stock", "cause_select_M2_DepotRelease": "Rolling_Stock",
    "cause_select_M3_TrainsetShortage": "Rolling_Stock",
    "cause_select_O1_Driving": "Eurostar_Operations", "cause_select_O2_TrainManager": "Eurostar_Operations",
    "cause_select_O3_Planning": "Eurostar_Operations", "cause_select_O4_OCC": "Eurostar_Operations",
    "cause_select_K1_Catering": "Eurostar_Operations", "cause_select_S4_TerminalStaff": "Eurostar_Operations",
    "cause_select_I3_InfraWorks": "Infrastructure", "cause_select_I4_InfraFault": "Infrastructure",
    "cause_select_S1_Passengers": "Passenger_Related", "cause_select_S2_Congestion": "Passenger_Related",
    "cause_select_I1_Animals": "Passenger_Related", "cause_select_I2_People": "Passenger_Related",
    "cause_select_I5_TrainsIncident": "Passenger_Related",
    "cause_select_I6_TrafficReg": "Traffic_Regulation", "cause_select_A4_LostPathway": "Traffic_Regulation",
    "cause_select_Weather_Safety": "Weather_Safety", "cause_select_Unknown": "Unknown",
    "cause_select_Other_rare_within_expanded": "Other_rare_within_expanded",
    "cause_select_Delayed_NoCauseRecord": "Delayed_NoCauseRecord",
}

In [26]:
coef_table = result_select.params[cause_select_dummies].rename("coef").to_frame()
coef_table["p_value"] = result_select.pvalues[cause_select_dummies]
coef_table["n"] = df[cause_select_dummies].sum()
coef_table["group"] = coef_table.index.map(select_to_group)
print("\n cause_select_* drill-down, grouped + sorted ")
print(coef_table.sort_values(["group", "coef"]).to_string())
 
print("\nDone. Review the printed diagnostics (Steps 1-2), the resid-vs-raw comparison "
      "(Step 6), and the VIF table (Step 7) before we lock this as the new baseline.")
 


 cause_select_* drill-down, grouped + sorted 
                                             coef  p_value      n                       group
cause_select_Delayed_NoCauseRecord      -0.103735      NaN  11023       Delayed_NoCauseRecord
cause_select_K1_Catering                -0.182254      NaN    547         Eurostar_Operations
cause_select_O2_TrainManager            -0.182191      NaN    573         Eurostar_Operations
cause_select_S4_TerminalStaff           -0.177994      NaN    999         Eurostar_Operations
cause_select_O1_Driving                 -0.119500      NaN   2005         Eurostar_Operations
cause_select_O4_OCC                     -0.093266      NaN   1063         Eurostar_Operations
cause_select_O3_Planning                -0.054956      NaN    512         Eurostar_Operations
cause_select_I4_InfraFault              -0.125170      NaN   8865              Infrastructure
cause_select_I3_InfraWorks               0.000068      NaN   1939              Infrastructure
cause_select_

In [27]:
#  helpers + one new import
from scipy.stats import chi2

def make_ladder_interactions(df, ladder, moderator, prefix):
    cols = []
    for label in ladder:
        col = f"{prefix}_x_{label}"
        df[col] = (df[label] * df[moderator]).astype(float)
        cols.append(col)
    return cols

def interaction_table(df, result, moderator_map):
    rows = []
    for moderator, cols in moderator_map.items():
        for col in cols:
            rows.append({
                "moderator": moderator, "term": col,
                "coef": result.params[col], "p_value": result.pvalues[col],
                "n_combo": int(df[col].sum()),
            })
    return pd.DataFrame(rows)

def lr_test(result_full, result_base, n_full, n_base, label):
    lr_stat = 2 * (result_full.llf - result_base.llf)
    dof = n_full - n_base
    p = chi2.sf(lr_stat, dof)
    print(f"{label}: LR={lr_stat:.2f}  df={dof}  p={p:.4g}")

In [28]:
# MODEL D: passenger-segment x arrival-delay breach ladder 
conn_int         = make_ladder_interactions(df, AD_LABELS, "has_connection", "conn")
nochoice_int     = make_ladder_interactions(df, AD_LABELS, "reason_no_choice", "nochoice")
preferstrain_int = make_ladder_interactions(df, AD_LABELS, "reason_prefers_train", "preferstrain")

segment_main_new = ["reason_no_choice", "reason_prefers_train"]  # has_connection already in core_vars_raw
core_vars_segment = core_vars_raw + segment_main_new + conn_int + nochoice_int + preferstrain_int

print("\n cell counts: breach x segment combos (flag any <30) ")
for mod, cols in [("has_connection", conn_int), ("reason_no_choice", nochoice_int), ("reason_prefers_train", preferstrain_int)]:
    print(mod, {c: int(df[c].sum()) for c in cols})

X_segment = df[core_vars_segment].astype(float)
model_segment = OrderedModel(y_cat, X_segment, distr="logit")
result_segment = model_segment.fit(method="bfgs", maxiter=500, disp=True)
print(result_segment.summary())
print("LL:", result_segment.llf, "AIC:", result_segment.aic)

lr_test(result_segment, result_raw, len(core_vars_segment), len(core_vars_raw), "Model D vs raw baseline")

seg_table = interaction_table(df, result_segment, {
    "has_connection": conn_int, "reason_no_choice": nochoice_int, "reason_prefers_train": preferstrain_int,
})
print("\n Model D interaction coefficients \n", seg_table.round(4).to_string(index=False))


 cell counts: breach x segment combos (flag any <30) 
has_connection {'conn_x_ad_b1_10.5': 5321, 'conn_x_ad_b2_19.5': 3889, 'conn_x_ad_b3_26.5': 3068, 'conn_x_ad_b4_46.5': 1662, 'conn_x_ad_b5_91.5': 504, 'conn_x_ad_b6_142.5': 180}
reason_no_choice {'nochoice_x_ad_b1_10.5': 1841, 'nochoice_x_ad_b2_19.5': 1455, 'nochoice_x_ad_b3_26.5': 1214, 'nochoice_x_ad_b4_46.5': 723, 'nochoice_x_ad_b5_91.5': 219, 'nochoice_x_ad_b6_142.5': 90}
reason_prefers_train {'preferstrain_x_ad_b1_10.5': 21801, 'preferstrain_x_ad_b2_19.5': 15710, 'preferstrain_x_ad_b3_26.5': 12076, 'preferstrain_x_ad_b4_46.5': 6384, 'preferstrain_x_ad_b5_91.5': 1889, 'preferstrain_x_ad_b6_142.5': 640}
Optimization terminated successfully.
         Current function value: 0.944600
         Iterations: 242
         Function evaluations: 252
         Gradient evaluations: 252
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood: 

In [29]:
# ── MODEL E: fleet (equip_RUB) x arrival-delay breach ladder 
# TGH is now the implicit Continental reference (dropped from main model),
# so rub_int tests whether RUB's delay-sensitivity slope differs from TGH's.
# tgh_int is omitted — TGH=reference means equip_TGH is always 0, causing perfect collinearity.
rub_int = make_ladder_interactions(df, AD_LABELS, "equip_RUB", "rub")
core_vars_fleet = core_vars_raw + rub_int 

print("\n cell counts: breach x fleet combos ")
print("equip_RUB", {c: int(df[c].sum()) for c in rub_int})

X_fleet = df[core_vars_fleet].astype(float)
model_fleet = OrderedModel(y_cat, X_fleet, distr="logit")
result_fleet = model_fleet.fit(method="bfgs", maxiter=500, disp=True)
print(result_fleet.summary())
print("LL:", result_fleet.llf, "AIC:", result_fleet.aic)

lr_test(result_fleet, result_raw, len(core_vars_fleet), len(core_vars_raw), "Model E vs raw baseline")

fleet_table = interaction_table(df, result_fleet, {"equip_RUB": rub_int})
print("\n Model E interaction coefficients \n", fleet_table.round(4).to_string(index=False))


X_vif_fleet = add_constant(X_fleet[["equip_RUB", "equip_E320", "is_channel"] + rub_int])
vif_fleet = pd.DataFrame({
    "variable": X_vif_fleet.columns,
    "VIF": [variance_inflation_factor(X_vif_fleet.values, i) for i in range(X_vif_fleet.shape[1])],
})
print("\n VIF (fleet block only) \n", vif_fleet.sort_values("VIF", ascending=False).to_string(index=False))


 cell counts: breach x fleet combos 
equip_RUB {'rub_x_ad_b1_10.5': 2780, 'rub_x_ad_b2_19.5': 1802, 'rub_x_ad_b3_26.5': 1361, 'rub_x_ad_b4_46.5': 671, 'rub_x_ad_b5_91.5': 240, 'rub_x_ad_b6_142.5': 83}
Optimization terminated successfully.
         Current function value: 0.955599
         Iterations: 228
         Function evaluations: 238
         Gradient evaluations: 238
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7046e+05
Model:                              OrderedModel   AIC:                         3.410e+05
Method:                       Maximum Likelihood   BIC:                         3.416e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   21:10:04                                         
No. Observations:                         178384                                   

In [30]:
# ── MODEL F: operational buffering + dual-delay compounding ──
df["clean_x_ad26"]        = (df["clean_score_routine_channel"] * df["ad_b3_26.5"]).astype(float)
df["dualdelay_ad26_ej31"] = (df["ad_b3_26.5"] * df["ej_b4_31.5"]).astype(float)
df["dualdelay_ad46_ej31"] = (df["ad_b4_46.5"] * df["ej_b4_31.5"]).astype(float)

buffer_vars = ["clean_x_ad26", "dualdelay_ad26_ej31", "dualdelay_ad46_ej31"]
core_vars_buffer = core_vars_raw + buffer_vars

print("\n cell counts / distribution: buffer & dual-delay terms ")
print("clean_x_ad26 (0 unless Channel & delay>26.5):")
print(df["clean_x_ad26"].describe())
print("\ndualdelay_ad26_ej31 (both breached): n =", int(df["dualdelay_ad26_ej31"].sum()))
print("dualdelay_ad46_ej31 (both breached): n =", int(df["dualdelay_ad46_ej31"].sum()))

X_buffer = df[core_vars_buffer].astype(float)
model_buffer = OrderedModel(y_cat, X_buffer, distr="logit")
result_buffer = model_buffer.fit(method="bfgs", maxiter=500, disp=True)
print(result_buffer.summary())
print("LL:", result_buffer.llf, "AIC:", result_buffer.aic)

lr_test(result_buffer, result_raw, len(core_vars_buffer), len(core_vars_raw), "Model F vs raw baseline")

print("\n Model F interaction coefficients \n",
      pd.DataFrame({
          "term": buffer_vars,
          "coef": [result_buffer.params[v] for v in buffer_vars],
          "p_value": [result_buffer.pvalues[v] for v in buffer_vars],
      }).round(4).to_string(index=False))


 cell counts / distribution: buffer & dual-delay terms 
clean_x_ad26 (0 unless Channel & delay>26.5):
count    178384.000000
mean          7.388085
std          25.311150
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: clean_x_ad26, dtype: float64

dualdelay_ad26_ej31 (both breached): n = 10226
dualdelay_ad46_ej31 (both breached): n = 8174
Optimization terminated successfully.
         Current function value: 0.955386
         Iterations: 210
         Function evaluations: 223
         Gradient evaluations: 223
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7043e+05
Model:                              OrderedModel   AIC:                         3.409e+05
Method:                       Maximum Likelihood   BIC:                         3.414e+05
Date:                           Sat, 20 Jun 2026 

In [31]:
# comparison summary 
print("\n interaction models: fit comparison vs raw baseline ")
print(f"{'spec':12s} {'n_vars':>8s} {'LL':>14s} {'AIC':>12s}")
print(f"{'raw (base)':12s} {len(core_vars_raw):8d} {result_raw.llf:14.2f} {result_raw.aic:12.2f}")
print(f"{'segment':12s} {len(core_vars_segment):8d} {result_segment.llf:14.2f} {result_segment.aic:12.2f}")
print(f"{'fleet':12s} {len(core_vars_fleet):8d} {result_fleet.llf:14.2f} {result_fleet.aic:12.2f}")
print(f"{'buffer':12s} {len(core_vars_buffer):8d} {result_buffer.llf:14.2f} {result_buffer.aic:12.2f}")


 interaction models: fit comparison vs raw baseline 
spec           n_vars             LL          AIC
raw (base)         44     -170475.92    341043.85
segment            64     -168501.60    337135.20
fleet              50     -170463.54    341031.07
buffer             47     -170425.49    340948.98


In [32]:
# clean_below_95: operationally-anchored (Eurostar KPI target = 95)
df["clean_below_95"] = pd.NA
df.loc[df["last_clean_score"].notna(), "clean_below_95"] = (
    df.loc[df["last_clean_score"].notna(), "last_clean_score"] < 95
).astype(int)

df["clean_below_95_channel"] = np.where(
    (df["is_channel"] == 1) & (df["clean_below_95"] == 1), 1, 0
).astype(float)

print("clean_below_95_channel distribution:")
print(df["clean_below_95_channel"].value_counts())
print("As % of Channel rows:", df.loc[df["is_channel"]==1, "clean_below_95_channel"].mean().round(3))
print("\nMean NPS by threshold (Channel only):")
print(df[df["is_channel"]==1].groupby("clean_below_95_channel")[TARGET].mean().round(3))

clean_below_95_channel distribution:
clean_below_95_channel
0.0    107874
1.0     70510
Name: count, dtype: int64
As % of Channel rows: 0.606

Mean NPS by threshold (Channel only):
clean_below_95_channel
0.0    8.230
1.0    8.254
Name: question_recommendation_nps_a, dtype: float64


In [33]:
# Model G
core_vars_g = [v if v != "clean_score_routine_channel" else "clean_below_95_channel"
               for v in core_vars_raw]
X_g = df[core_vars_g].astype(float)
model_g = OrderedModel(y_cat, X_g, distr="logit")
result_g = model_g.fit(method="bfgs", maxiter=500, disp=True)
print(result_g.summary())
print("LL:", result_g.llf, "AIC:", result_g.aic)
lr_test(result_g, result_raw, len(core_vars_g), len(core_vars_raw),
        "Model G (below-95 threshold) vs raw baseline")
print("\nclean_below_95_channel coef:", round(result_g.params["clean_below_95_channel"], 4),
      "  p:", round(result_g.pvalues["clean_below_95_channel"], 4))


Optimization terminated successfully.
         Current function value: 0.955667
         Iterations: 210
         Function evaluations: 219
         Gradient evaluations: 219
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7048e+05
Model:                              OrderedModel   AIC:                         3.410e+05
Method:                       Maximum Likelihood   BIC:                         3.415e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   21:59:45                                         
No. Observations:                         178384                                         
Df Residuals:                             178338                                         
Df Model:                                     44                                         
               

In [34]:
# ── MODEL H ──
df["below95_x_ad26"] = (df["clean_below_95_channel"] * df["ad_b3_26.5"]).astype(float)
df["below95_x_ad46"] = (df["clean_below_95_channel"] * df["ad_b4_46.5"]).astype(float)
core_vars_h = core_vars_g + ["below95_x_ad26", "below95_x_ad46"]

print("\ncell counts (below KPI target AND delay breached):")
print("below_95 & delay>26.5:", int(df["below95_x_ad26"].sum()))
print("below_95 & delay>46.5:", int(df["below95_x_ad46"].sum()))

X_h = df[core_vars_h].astype(float)
model_h = OrderedModel(y_cat, X_h, distr="logit")
result_h = model_h.fit(method="bfgs", maxiter=500, disp=True)
print(result_h.summary())
print("LL:", result_h.llf, "AIC:", result_h.aic)
lr_test(result_h, result_g, len(core_vars_h), len(core_vars_g),
        "Model H (+ interaction) vs Model G")
lr_test(result_h, result_raw, len(core_vars_h), len(core_vars_raw),
        "Model H vs raw baseline")
print("\nModel H interaction terms:")
for v in ["clean_below_95_channel", "below95_x_ad26", "below95_x_ad46"]:
    print(f"  {v}: coef={result_h.params[v]:.4f}  p={result_h.pvalues[v]:.4f}")



cell counts (below KPI target AND delay breached):
below_95 & delay>26.5: 8526
below_95 & delay>46.5: 4606
Optimization terminated successfully.
         Current function value: 0.955513
         Iterations: 211
         Function evaluations: 220
         Gradient evaluations: 220
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.7045e+05
Model:                              OrderedModel   AIC:                         3.410e+05
Method:                       Maximum Likelihood   BIC:                         3.415e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   22:18:19                                         
No. Observations:                         178384                                         
Df Residuals:                             178336                                       

In [35]:

print("\n all interaction models: fit comparison vs raw baseline ")
print(f"{'spec':15s} {'n_vars':>8s} {'LL':>14s} {'AIC':>12s}")
for label, res, nvars in [
    ("raw (base)",        result_raw,     len(core_vars_raw)),
    ("segment (D)",       result_segment, len(core_vars_segment)),
    ("fleet (E)",         result_fleet,   len(core_vars_fleet)),
    ("buffer (F)",        result_buffer,  len(core_vars_buffer)),
    ("below95 (G)",       result_g,       len(core_vars_g)),
    ("below95×delay (H)", result_h,       len(core_vars_h)),
]:
    print(f"{label:15s} {nvars:8d} {res.llf:14.2f} {res.aic:12.2f}")


 all interaction models: fit comparison vs raw baseline 
spec              n_vars             LL          AIC
raw (base)            44     -170475.92    341043.85
segment (D)           64     -168501.60    337135.20
fleet (E)             50     -170463.54    341031.07
buffer (F)            47     -170425.49    340948.98
below95 (G)           44     -170475.62    341043.23
below95×delay (H)       46     -170448.18    340992.35


In [36]:

from scipy.special import expit

def predict_manual(params, x_vals):
    
    n_coef = len(params) - 2
    beta = params[:n_coef]
    t0 = params[n_coef]
    t1 = t0 + np.exp(params[n_coef + 1])   # second threshold = t0 + exp(log_delta)
    xb = np.dot(x_vals, beta)
    p_det  = expit(t0 - xb)                # P(Detractor)
    p_pass = expit(t1 - xb) - p_det        # P(Passive)
    p_prom = 1 - p_det - p_pass            # P(Promoter)
    return (p_prom - p_det) * 100          # NPS point

params_orig = result_raw.params.values.copy()
cov         = result_raw.cov_params().values
draws       = np.random.multivariate_normal(params_orig, cov, size=2000)

base_vals = X_raw.mean().values  # shape (n_features,)
col_index = {c: i for i, c in enumerate(X_raw.columns)}

delay_scenarios = {
    "no_delay":   [],
    ">10.5 min":  ["ad_b1_10.5"],
    ">19.5 min":  ["ad_b1_10.5","ad_b2_19.5"],
    ">26.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5"],
    ">46.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5"],
    ">91.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5"],
    ">142.5 min": ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5","ad_b6_142.5"],
}

rows = []
for label, breach_cols in delay_scenarios.items():
    x = base_vals.copy()
    for c in breach_cols:
        x[col_index[c]] = 1
    for c in ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5","ad_b6_142.5"]:
        x[col_index[c]] = 1 if c in breach_cols else 0

    point = predict_manual(params_orig, x)
    ci    = [predict_manual(d, x) for d in draws]
    rows.append({
        "scenario":  label,
        "nps_point": round(point, 2),
        "ci_lower":  round(np.percentile(ci, 2.5), 2),
        "ci_upper":  round(np.percentile(ci, 97.5), 2),
    })

result_df = pd.DataFrame(rows)
result_df["nps_change"] = (result_df["nps_point"] - result_df.loc[0,"nps_point"]).round(2)
print(result_df.to_string(index=False))

  scenario  nps_point  ci_lower  ci_upper  nps_change
  no_delay      41.95     41.52     42.39        0.00
 >10.5 min      34.65     33.31     35.98       -7.30
 >19.5 min      28.33     26.53     30.14      -13.62
 >26.5 min      16.62     14.93     18.17      -25.33
 >46.5 min      -4.84     -6.77     -2.99      -46.79
 >91.5 min     -22.46    -25.64    -19.31      -64.41
>142.5 min     -39.43    -43.56    -35.35      -81.38


In [37]:
import numpy as np
import pandas as pd
from scipy.special import expit


def predict_manual(params, x_vals):
    n_coef = len(params) - 2
    beta   = params[:n_coef]
    t0     = params[n_coef]
    t1     = t0 + np.exp(params[n_coef + 1])
    xb     = np.dot(x_vals, beta)
    p_det  = expit(t0 - xb)
    p_prom = 1 - expit(t1 - xb)
    return (p_prom - p_det) * 100

In [38]:
params_raw    = result_raw.params.values.copy()
cov_raw       = result_raw.cov_params().values
draws_raw     = np.random.multivariate_normal(params_raw, cov_raw, size=2000)
col_idx_raw   = {c: i for i, c in enumerate(X_raw.columns)}
base_raw      = X_raw.mean().values

all_delay_cols    = ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5","ad_b6_142.5",
                     "ej_b1_10.5","ej_b2_13.5","ej_b3_20.5","ej_b4_31.5"]
all_cause_cols    = ["cause_Eurostar_Operations","cause_Infrastructure","cause_Passenger_Related",
                     "cause_Rolling_Stock","cause_Traffic_Regulation","cause_Weather_Safety",
                     "cause_Delayed_NoCauseRecord"]
zero_base_raw     = {c: 0 for c in all_delay_cols + all_cause_cols}


In [39]:
def nps_raw(overrides_on, overrides_off=None):
    x = base_raw.copy()
    if overrides_off:
        for col, val in overrides_off.items():
            if col in col_idx_raw: x[col_idx_raw[col]] = val
    for col, val in overrides_on.items():
        if col in col_idx_raw: x[col_idx_raw[col]] = val
    pt = predict_manual(params_raw, x)
    ci = [predict_manual(d, x) for d in draws_raw]
    return round(pt,2), round(np.percentile(ci,2.5),2), round(np.percentile(ci,97.5),2)


In [40]:
base_nps_raw, _, _ = nps_raw(zero_base_raw)
print(f"Raw model base NPS: {base_nps_raw:.2f}")

Raw model base NPS: 46.33


In [41]:
delay_scenarios = {
    "no_delay":   [],
    ">10.5 min":  ["ad_b1_10.5"],
    ">19.5 min":  ["ad_b1_10.5","ad_b2_19.5"],
    ">26.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5"],
    ">46.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5"],
    ">91.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5"],
    ">142.5 min": ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5","ad_b6_142.5"],
}

In [42]:
ladder_rows = []
for label, breach_cols in delay_scenarios.items():
    overrides = {**zero_base_raw}
    for c in breach_cols: overrides[c] = 1
    pt, lo, hi = nps_raw(overrides)
    ladder_rows.append({"scenario": label, "nps_point": pt, "ci_lower": lo,
                        "ci_upper": hi, "nps_change": round(pt - base_nps_raw, 2)})
ladder_df = pd.DataFrame(ladder_rows)
print("\n ARRIVAL DELAY LADDER ")
print(ladder_df.to_string(index=False))


 ARRIVAL DELAY LADDER 
  scenario  nps_point  ci_lower  ci_upper  nps_change
  no_delay      46.33     45.68     47.01        0.00
 >10.5 min      39.29     37.82     40.79       -7.04
 >19.5 min      33.15     31.15     35.13      -13.18
 >26.5 min      21.67     19.79     23.59      -24.66
 >46.5 min       0.34     -1.91      2.63      -45.99
 >91.5 min     -17.43    -20.93    -14.14      -63.76
>142.5 min     -34.80    -39.00    -30.68      -81.13


In [43]:
ej_scenarios = {
    "no_delay":   [],
    ">10.5 min":  ["ej_b1_10.5"],
    ">20.5 min":  ["ej_b1_10.5","ej_b2_13.5","ej_b3_20.5"],
    ">31.5 min":  ["ej_b1_10.5","ej_b2_13.5","ej_b3_20.5","ej_b4_31.5"],
}
ej_rows = []
for label, breach_cols in ej_scenarios.items():
    overrides = {**zero_base_raw}
    for c in breach_cols: overrides[c] = 1
    pt, lo, hi = nps_raw(overrides)
    ej_rows.append({"scenario": label, "nps_point": pt, "ci_lower": lo,
                    "ci_upper": hi, "nps_change": round(pt - base_nps_raw, 2)})
ej_df = pd.DataFrame(ej_rows)
print("\n EARLY-JOURNEY DELAY ")
print(ej_df.to_string(index=False))


 EARLY-JOURNEY DELAY 
 scenario  nps_point  ci_lower  ci_upper  nps_change
 no_delay      46.33     45.68     47.01        0.00
>10.5 min      38.87     36.12     41.51       -7.46
>20.5 min      32.02     29.82     34.23      -14.31
>31.5 min      26.64     24.33     28.78      -19.69


In [44]:
ctrl_scenarios = [
    ("Equipment change",                 {"has_equipment_change":1},  {"has_equipment_change":0}),
    ("Fleet: RUB vs TGH (Continental)", {"equip_RUB":1,"equip_E320":0,"is_channel":0},
                                        {"equip_RUB":0,"equip_E320":0,"is_channel":0}),
    ("Fleet: E320 vs E300 (Channel)",   {"equip_E320":1,"equip_RUB":0,"is_channel":1},
                                        {"equip_E320":0,"equip_RUB":0,"is_channel":1}),
    ("Occupancy >400 pax",              {"total_pax_breach_400":1},  {"total_pax_breach_400":0}),
    ("Has connection",                  {"has_connection":1},         {"has_connection":0}),
    ("Channel vs Continental",          {"is_channel":1},             {"is_channel":0}),
    ("Price +£50 from mean",            {"metadata_price": X_raw["metadata_price"].mean()+50}, {}),
    ("Price +£100 from mean",           {"metadata_price": X_raw["metadata_price"].mean()+100}, {}),
]

ctrl_rows = []
for label, scen_vals, base_vals_override in ctrl_scenarios:
    base_overrides = {**zero_base_raw, **base_vals_override}
    scen_overrides = {**zero_base_raw, **base_vals_override, **scen_vals}
    base_pt, _, _   = nps_raw(base_overrides)
    scen_pt, lo, hi = nps_raw(scen_overrides)
    ctrl_rows.append({"variable": label, "baseline_nps": base_pt, "scenario_nps": scen_pt,
                      "nps_change": round(scen_pt - base_pt, 2),
                      "ci_lower": round(lo - base_pt, 2), "ci_upper": round(hi - base_pt, 2)})

ctrl_df = pd.DataFrame(ctrl_rows)
print("\n CONTROLLABLE VARIABLES ")
print(ctrl_df.sort_values("nps_change").to_string(index=False))



 CONTROLLABLE VARIABLES 
                       variable  baseline_nps  scenario_nps  nps_change  ci_lower  ci_upper
               Equipment change         46.80         37.07       -9.73    -11.50     -8.01
          Price +£100 from mean         46.33         37.29       -9.04     -9.96     -8.09
           Price +£50 from mean         46.33         41.90       -4.43     -5.18     -3.68
             Occupancy >400 pax         46.65         42.45       -4.20     -5.54     -2.87
  Fleet: E320 vs E300 (Channel)         53.43         50.86       -2.57     -3.89     -1.31
                 Has connection         46.56         44.37       -2.19     -3.31     -1.00
Fleet: RUB vs TGH (Continental)         34.97         46.70       11.73      8.86     14.48
         Channel vs Continental         33.83         52.47       18.64     17.37     19.86


In [45]:
cause_cols_map = {
    "Rolling_Stock":         "cause_Rolling_Stock",
    "Passenger_Related":     "cause_Passenger_Related",
    "Eurostar_Operations":   "cause_Eurostar_Operations",
    "Delayed_NoCauseRecord": "cause_Delayed_NoCauseRecord",
    "Infrastructure":        "cause_Infrastructure",
    "Traffic_Regulation":    "cause_Traffic_Regulation",
    "Weather_Safety":        "cause_Weather_Safety",
}

In [46]:
cause_rows = []
for label, col in cause_cols_map.items():
    for delay_label, delay_set in [("no_delay",{}),(">26.5 min",{"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1}),
                                    (">46.5 min",{"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1,"ad_b4_46.5":1})]:
        overrides = {**zero_base_raw, col: 1, **delay_set}
        pt, lo, hi = nps_raw(overrides)
        cause_rows.append({"cause": label, "delay_band": delay_label, "nps_point": pt,
                           "ci_lower": lo, "ci_upper": hi, "vs_base": round(pt - base_nps_raw, 2)})
cause_df = pd.DataFrame(cause_rows)
print("\n GROUP CAUSE RANKING at >26.5 min")
at26 = cause_df[cause_df.delay_band==">26.5 min"].sort_values("vs_base")
print(at26[["cause","nps_point","ci_lower","ci_upper","vs_base"]].to_string(index=False))




 GROUP CAUSE RANKING at >26.5 min
                cause  nps_point  ci_lower  ci_upper  vs_base
        Rolling_Stock      11.04      8.88     13.25   -35.29
    Passenger_Related      14.71     12.89     16.52   -31.62
  Eurostar_Operations      16.29     13.77     18.83   -30.04
Delayed_NoCauseRecord      17.37     15.21     19.54   -28.96
       Infrastructure      17.40     15.41     19.54   -28.93
   Traffic_Regulation      19.66     17.80     21.48   -26.67
       Weather_Safety      20.18     17.45     23.02   -26.15


In [47]:
dual = {
    "No delay":           {},
    "Arrival >26.5 only": {"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1},
    "EJ >31.5 only":      {"ej_b1_10.5":1,"ej_b2_13.5":1,"ej_b3_20.5":1,"ej_b4_31.5":1},
    "Both breached":      {"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1,
                           "ej_b1_10.5":1,"ej_b2_13.5":1,"ej_b3_20.5":1,"ej_b4_31.5":1},
}

In [48]:
dual_rows = []
for label, delay_set in dual.items():
    overrides = {**zero_base_raw, **delay_set}
    pt, lo, hi = nps_raw(overrides)
    dual_rows.append({"scenario": label, "nps_point": pt, "ci_lower": lo, "ci_upper": hi})
dual_df = pd.DataFrame(dual_rows)
ad_only = dual_df.loc[dual_df.scenario=="Arrival >26.5 only","nps_point"].values[0]
ej_only = dual_df.loc[dual_df.scenario=="EJ >31.5 only","nps_point"].values[0]
both    = dual_df.loc[dual_df.scenario=="Both breached","nps_point"].values[0]
additive = base_nps_raw + (ad_only - base_nps_raw) + (ej_only - base_nps_raw)
print("\n DUAL-DELAY COMPOUNDING ")
print(dual_df.to_string(index=False))
print(f"\nAdditive expectation: {additive:.2f}  |  Actual both-breached: {both:.2f}  |  Compounding penalty: {both - additive:.2f}")



 DUAL-DELAY COMPOUNDING 
          scenario  nps_point  ci_lower  ci_upper
          No delay      46.33     45.68     47.01
Arrival >26.5 only      21.67     19.79     23.59
     EJ >31.5 only      26.64     24.33     28.78
     Both breached      -0.15     -2.80      2.35

Additive expectation: 1.98  |  Actual both-breached: -0.15  |  Compounding penalty: -2.13


In [49]:
params_sel  = result_select.params.values.copy()
col_idx_sel = {c: i for i, c in enumerate(X_select.columns)}
base_sel    = X_select.mean().values
zero_base_sel = {c: 0 for c in all_delay_cols + cause_select_dummies}

In [50]:
def nps_sel(overrides):
    x = base_sel.copy()
    for col, val in overrides.items():
        if col in col_idx_sel:
            x[col_idx_sel[col]] = val
    return round(predict_manual(params_sel, x), 2)

base_nps_sel = nps_sel(zero_base_sel)
print(f"Granular model base NPS: {base_nps_sel:.2f}")

Granular model base NPS: 46.33


In [51]:
select_to_group = {
    "cause_select_M1_Reliability":"Rolling_Stock","cause_select_M2_DepotRelease":"Rolling_Stock",
    "cause_select_M3_TrainsetShortage":"Rolling_Stock",
    "cause_select_O1_Driving":"Eurostar_Operations","cause_select_O2_TrainManager":"Eurostar_Operations",
    "cause_select_O3_Planning":"Eurostar_Operations","cause_select_O4_OCC":"Eurostar_Operations",
    "cause_select_K1_Catering":"Eurostar_Operations","cause_select_S4_TerminalStaff":"Eurostar_Operations",
    "cause_select_I3_InfraWorks":"Infrastructure","cause_select_I4_InfraFault":"Infrastructure",
    "cause_select_S1_Passengers":"Passenger_Related","cause_select_S2_Congestion":"Passenger_Related",
    "cause_select_I1_Animals":"Passenger_Related","cause_select_I2_People":"Passenger_Related",
    "cause_select_I5_TrainsIncident":"Passenger_Related",
    "cause_select_I6_TrafficReg":"Traffic_Regulation","cause_select_A4_LostPathway":"Traffic_Regulation",
    "cause_select_Weather_Safety":"Weather_Safety","cause_select_Delayed_NoCauseRecord":"Delayed_NoCauseRecord",
    "cause_select_Unknown":"Unknown","cause_select_Other_rare_within_expanded":"Other",
}

In [52]:
sel_rows = []
for col in cause_select_dummies:
    n    = int(df[col].sum())
    coef = float(result_select.params.get(col, np.nan))
    for delay_label, delay_set in [
        ("no_delay",  {}),
        (">26.5 min", {"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1}),
        (">46.5 min", {"ad_b1_10.5":1,"ad_b2_19.5":1,"ad_b3_26.5":1,"ad_b4_46.5":1}),
    ]:
        overrides = {**zero_base_sel, col: 1, **delay_set}
        pt = nps_sel(overrides)
        sel_rows.append({
            "group":      select_to_group.get(col, "Other"),
            "cause":      col.replace("cause_select_", ""),
            "n":          n,
            "coef":       round(coef, 3),
            "delay_band": delay_label,
            "nps_point":  pt,
            "vs_base":    round(pt - base_nps_sel, 2),
        })


In [53]:
sel_df = pd.DataFrame(sel_rows)

for band in [">26.5 min", ">46.5 min"]:
    subset = sel_df[sel_df.delay_band == band].sort_values(["group","vs_base"])
    print(f"\n══ GRANULAR CAUSE NPS IMPACT at {band} (point estimates, no CI) ══")
    print(f"{'Group':<28} {'Cause':<32} {'n':>6}  {'coef':>7}  {'NPS':>7}  {'vs_base':>8}")
    print("─" * 95)
    prev = None
    for _, r in subset.iterrows():
        if r.group != prev: print(); prev = r.group
        print(f"{r.group:<28} {r.cause:<32} {r.n:>6}  {r.coef:>7.3f}  {r.nps_point:>7.2f}  {r.vs_base:>8.2f}")



══ GRANULAR CAUSE NPS IMPACT at >26.5 min (point estimates, no CI) ══
Group                        Cause                                 n     coef      NPS   vs_base
───────────────────────────────────────────────────────────────────────────────────────────────

Delayed_NoCauseRecord        Delayed_NoCauseRecord             11023   -0.104    16.67    -29.66

Eurostar_Operations          K1_Catering                         547   -0.182    13.45    -32.88
Eurostar_Operations          O2_TrainManager                     573   -0.182    13.45    -32.88
Eurostar_Operations          S4_TerminalStaff                    999   -0.178    13.62    -32.71
Eurostar_Operations          O1_Driving                         2005   -0.120    16.02    -30.31
Eurostar_Operations          O4_OCC                             1063   -0.093    17.10    -29.23
Eurostar_Operations          O3_Planning                         512   -0.055    18.66    -27.67

Infrastructure               I4_InfraFault            

In [54]:
print("\n WORST CAUSE PER GROUP at >26.5 min ")
worst = (sel_df[sel_df.delay_band==">26.5 min"]
         .loc[sel_df[sel_df.delay_band==">26.5 min"].groupby("group")["vs_base"].idxmin(),
              ["group","cause","n","coef","nps_point","vs_base"]]
         .sort_values("vs_base"))
print(worst.to_string(index=False))


 WORST CAUSE PER GROUP at >26.5 min 
                group                      cause     n   coef  nps_point  vs_base
        Rolling_Stock        M3_TrainsetShortage  1391 -0.346       6.64   -39.69
    Passenger_Related              S2_Congestion  3982 -0.251      10.59   -35.74
              Unknown                    Unknown    93 -0.200      12.72   -33.61
  Eurostar_Operations                K1_Catering   547 -0.182      13.45   -32.88
       Infrastructure              I4_InfraFault  8865 -0.125      15.79   -30.54
Delayed_NoCauseRecord      Delayed_NoCauseRecord 11023 -0.104      16.67   -29.66
   Traffic_Regulation              I6_TrafficReg 57827 -0.049      18.89   -27.44
       Weather_Safety             Weather_Safety  3860 -0.030      19.66   -26.67
                Other Other_rare_within_expanded   526  0.033      22.22   -24.11


In [55]:
sel_df.to_csv("nps_cause_select.csv", index=False)
print("\nSaved: nps_cause_select.csv")



Saved: nps_cause_select.csv


In [56]:

ladder_df.to_csv("nps_delay_ladder.csv", index=False)
ctrl_df.to_csv("nps_controllable_vars.csv", index=False)
cause_df.to_csv("nps_cause_group_ranking.csv", index=False)


In [57]:

core_vars_staff = core_vars_raw + ["staff_composite"]
X_staff = df[core_vars_staff].astype(float)
result_staff = OrderedModel(y_cat, X_staff, distr="logit").fit(method="bfgs", maxiter=500)
print(result_staff.summary())
lr_test(result_staff, result_raw, len(core_vars_staff), len(core_vars_raw), "Staff mediation vs base")


Optimization terminated successfully.
         Current function value: 0.889834
         Iterations: 212
         Function evaluations: 222
         Gradient evaluations: 222
                                   OrderedModel Results                                  
Dep. Variable:     question_recommendation_nps_a   Log-Likelihood:            -1.5873e+05
Model:                              OrderedModel   AIC:                         3.176e+05
Method:                       Maximum Likelihood   BIC:                         3.180e+05
Date:                           Sat, 20 Jun 2026                                         
Time:                                   22:38:05                                         
No. Observations:                         178384                                         
Df Residuals:                             178337                                         
Df Model:                                     45                                         
               

In [58]:
from scipy.special import expit
import numpy as np

def predict_full(params, x_vals):
    n_coef = len(params) - 2
    beta = params[:n_coef]
    t0 = params[n_coef]
    t1 = t0 + np.exp(params[n_coef + 1])
    xb = np.dot(x_vals, beta)
    p_det  = expit(t0 - xb)
    p_prom = 1 - expit(t1 - xb)
    p_pass = 1 - p_det - p_prom
    nps    = (p_prom - p_det) * 100
    return p_det, p_pass, p_prom, nps

params_raw = result_raw.params.values.copy()
cov_raw    = result_raw.cov_params().values
draws_raw  = np.random.multivariate_normal(params_raw, cov_raw, size=2000)
col_idx_raw = {c: i for i, c in enumerate(X_raw.columns)}
base_raw    = X_raw.mean().values



In [59]:
delay_scenarios = {
    "no_delay":   [],
    ">10.5 min":  ["ad_b1_10.5"],
    ">19.5 min":  ["ad_b1_10.5","ad_b2_19.5"],
    ">26.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5"],
    ">46.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5"],
    ">91.5 min":  ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5"],
    ">142.5 min": ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5","ad_b4_46.5","ad_b5_91.5","ad_b6_142.5"],
}

zero_base_raw = {c: 0 for c in ["ad_b1_10.5","ad_b2_19.5","ad_b3_26.5",
                                  "ad_b4_46.5","ad_b5_91.5","ad_b6_142.5",
                                  "ej_b1_10.5","ej_b2_13.5","ej_b3_20.5","ej_b4_31.5",
                                  "cause_Eurostar_Operations","cause_Infrastructure",
                                  "cause_Passenger_Related","cause_Rolling_Stock",
                                  "cause_Traffic_Regulation","cause_Weather_Safety",
                                  "cause_Delayed_NoCauseRecord"]}


In [60]:

rows = []
for label, breach_cols in delay_scenarios.items():
    x = base_raw.copy()
    for col, val in zero_base_raw.items():
        x[col_idx_raw[col]] = val
    for c in breach_cols:
        x[col_idx_raw[c]] = 1

    det, pas, pro, nps = predict_full(params_raw, x)
    

    det_ci  = [predict_full(d, x)[0] for d in draws_raw]
    pas_ci  = [predict_full(d, x)[1] for d in draws_raw]
    pro_ci  = [predict_full(d, x)[2] for d in draws_raw]
    

In [61]:
rows.append({
        "scenario":  label,
        "p_det":     round(det*100, 1),
        "det_lo":    round(np.percentile(det_ci, 2.5)*100, 1),
        "det_hi":    round(np.percentile(det_ci, 97.5)*100, 1),
        "p_pass":    round(pas*100, 1),
        "pas_lo":    round(np.percentile(pas_ci, 2.5)*100, 1),
        "pas_hi":    round(np.percentile(pas_ci, 97.5)*100, 1),
        "p_prom":    round(pro*100, 1),
        "pro_lo":    round(np.percentile(pro_ci, 2.5)*100, 1),
        "pro_hi":    round(np.percentile(pro_ci, 97.5)*100, 1),
        "nps":       round(nps, 1),
    })

class_df = pd.DataFrame(rows)
print(class_df.to_string(index=False))
class_df.to_csv("nps_class_breakdown.csv", index=False)

  scenario  p_det  det_lo  det_hi  p_pass  pas_lo  pas_hi  p_prom  pro_lo  pro_hi   nps
>142.5 min   50.0    47.2    52.8    34.8    33.4    36.2    15.2    13.8    16.7 -34.8


In [62]:
import time

In [63]:
core_vars_select_raw = (
    delay_dummies
    + cause_select_dummies
    + other_operational_fixed
    + RAW_BLOCK
    + ["metadata_price"]
)

In [64]:
X_sel_raw = df[core_vars_select_raw].astype(float)
print(f"Fitting point-estimate model ({len(core_vars_select_raw)} vars)...")
t0 = time.time()
result_sel_raw = OrderedModel(y_cat, X_sel_raw, distr="logit").fit(
    method="bfgs", maxiter=500, disp=True
)
print(f"Done in {time.time()-t0:.0f}s")


Fitting point-estimate model (58 vars)...
Optimization terminated successfully.
         Current function value: 0.955517
         Iterations: 200
         Function evaluations: 211
         Gradient evaluations: 211
Done in 2233s


In [65]:
point_coefs = result_sel_raw.params[cause_select_dummies]
print("\nPoint estimates (cause_select_* only):")
print(point_coefs.round(4).to_string())


Point estimates (cause_select_* only):
cause_select_A4_LostPathway                0.0733
cause_select_I1_Animals                    0.0651
cause_select_I2_People                    -0.1431
cause_select_I3_InfraWorks                 0.0021
cause_select_I4_InfraFault                -0.1249
cause_select_I5_TrainsIncident            -0.1297
cause_select_I6_TrafficReg                -0.0486
cause_select_K1_Catering                  -0.1779
cause_select_M1_Reliability               -0.2425
cause_select_M2_DepotRelease              -0.2399
cause_select_M3_TrainsetShortage          -0.3461
cause_select_O1_Driving                   -0.1209
cause_select_O2_TrainManager              -0.1797
cause_select_O3_Planning                  -0.0488
cause_select_O4_OCC                       -0.0922
cause_select_Other_rare_within_expanded    0.0381
cause_select_S1_Passengers                -0.2152
cause_select_S2_Congestion                -0.2510
cause_select_S4_TerminalStaff             -0.1770
cause_sele

In [72]:
print(result.params.index.tolist()[:30])

['ad_b1_10.5', 'ad_b2_19.5', 'ad_b3_26.5', 'ad_b4_46.5', 'ad_b5_91.5', 'ad_b6_142.5', 'ej_b1_10.5', 'ej_b2_13.5', 'ej_b3_20.5', 'ej_b4_31.5', 'cause_select_A4_LostPathway', 'cause_select_I1_Animals', 'cause_select_I2_People', 'cause_select_I3_InfraWorks', 'cause_select_I4_InfraFault', 'cause_select_I5_TrainsIncident', 'cause_select_I6_TrafficReg', 'cause_select_K1_Catering', 'cause_select_M1_Reliability', 'cause_select_M2_DepotRelease', 'cause_select_M3_TrainsetShortage', 'cause_select_O1_Driving', 'cause_select_O2_TrainManager', 'cause_select_O3_Planning', 'cause_select_O4_OCC', 'cause_select_Other_rare_within_expanded', 'cause_select_S1_Passengers', 'cause_select_S2_Congestion', 'cause_select_S4_TerminalStaff', 'cause_select_Unknown']


In [73]:
from scipy.stats import multivariate_normal
import numpy as np, pandas as pd


N_BOOT  = 2000          
result  = result_sel_raw    

mu    = result.params.values
sigma = result.cov_params().values


assert np.all(np.isfinite(sigma)), "cov_params has NaNs — switch to result_raw"

rng         = np.random.default_rng(42)
param_draws = rng.multivariate_normal(mu, sigma, size=N_BOOT)  # (2000, n_params)

cause_select_dummies = [c for c in result.params.index if c.startswith("cause_select_")]
col_idx = [list(result.params.index).index(c) for c in cause_select_dummies]

boot_arr = param_draws[:, col_idx]   # (2000, n_causes) — instant


point_coefs = result.params[cause_select_dummies]

ci_df = pd.DataFrame({
    "cause":    cause_select_dummies,
    "group":    [select_to_group.get(c, "Other") for c in cause_select_dummies],
    "n":        df[cause_select_dummies].sum().astype(int).values,
    "coef":     point_coefs.values,
    "ci_lower": np.percentile(boot_arr, 2.5,  axis=0),
    "ci_upper": np.percentile(boot_arr, 97.5, axis=0),
})

ci_df["cause_short"] = ci_df["cause"].str.replace("cause_select_", "")
ci_df["sig"]         = ~((ci_df["ci_lower"] < 0) & (ci_df["ci_upper"] > 0))
ci_df = ci_df.sort_values(["group", "coef"])

print("\nGRANULAR CAUSE CIs (parametric bootstrap, 95%)")
print(ci_df[["cause_short","group","n","coef","ci_lower","ci_upper","sig"]].to_string(index=False))


GRANULAR CAUSE CIs (parametric bootstrap, 95%)
               cause_short                 group     n      coef  ci_lower  ci_upper   sig
     Delayed_NoCauseRecord Delayed_NoCauseRecord 11023 -0.102949 -0.148282 -0.057655  True
           O2_TrainManager   Eurostar_Operations   573 -0.179717 -0.336854 -0.029502  True
               K1_Catering   Eurostar_Operations   547 -0.177904 -0.332332 -0.017110  True
          S4_TerminalStaff   Eurostar_Operations   999 -0.177044 -0.307344 -0.054289  True
                O1_Driving   Eurostar_Operations  2005 -0.120894 -0.207552 -0.039324  True
                    O4_OCC   Eurostar_Operations  1063 -0.092172 -0.211985  0.020801 False
               O3_Planning   Eurostar_Operations   512 -0.048846 -0.211443  0.125932 False
             I4_InfraFault        Infrastructure  8865 -0.124898 -0.176424 -0.075355  True
             I3_InfraWorks        Infrastructure  1939  0.002096 -0.086385  0.094979 False
Other_rare_within_expanded                

In [75]:
print("boot_coefs length:", len(boot_coefs))

if len(boot_coefs) == 0:
    # Re-run one iteration manually to surface the actual error
    import traceback
    try:
        cov_sel = result_sel_raw.cov_params().values
        params_sel_pt = result_sel_raw.params.values.copy()
        d = np.random.multivariate_normal(params_sel_pt, cov_sel)
        # replicate exactly what the loop body does:
        test = d[[col_idx_sel[c] for c in cause_select_dummies]]
        print("Single draw succeeded, shape:", test.shape)
    except Exception as e:
        traceback.print_exc()

boot_coefs length: 0
Single draw succeeded, shape: (22,)


In [76]:
ci_df["cause_short"] = ci_df["cause"].str.replace("cause_select_", "")
ci_df["sig"] = ~((ci_df["ci_lower"] < 0) & (ci_df["ci_upper"] > 0))  
ci_df = ci_df.sort_values(["group", "coef"])

print("\n GRANULAR CAUSE CIs (bootstrap 95%, sorted by group then severity) ")
print(f"{'Group':<28} {'Cause':<30} {'n':>6}  {'coef':>7}  {'CI lower':>9}  {'CI upper':>9}  {'sig':>5}")
print("─" * 100)
prev = None
for _, r in ci_df.iterrows():
    if r.group != prev:
        print()
        prev = r.group
    sig_str = "YES" if r.sig else "n.s."
    print(f"{r.group:<28} {r.cause_short:<30} {r.n:>6}  {r.coef:>7.4f}  "
          f"{r.ci_lower:>9.4f}  {r.ci_upper:>9.4f}  {sig_str:>5}")



 GRANULAR CAUSE CIs (bootstrap 95%, sorted by group then severity) 
Group                        Cause                               n     coef   CI lower   CI upper    sig
────────────────────────────────────────────────────────────────────────────────────────────────────

Delayed_NoCauseRecord        Delayed_NoCauseRecord           11023  -0.1029    -0.1483    -0.0577    YES

Eurostar_Operations          O2_TrainManager                   573  -0.1797    -0.3369    -0.0295    YES
Eurostar_Operations          K1_Catering                       547  -0.1779    -0.3323    -0.0171    YES
Eurostar_Operations          S4_TerminalStaff                  999  -0.1770    -0.3073    -0.0543    YES
Eurostar_Operations          O1_Driving                       2005  -0.1209    -0.2076    -0.0393    YES
Eurostar_Operations          O4_OCC                           1063  -0.0922    -0.2120     0.0208   n.s.
Eurostar_Operations          O3_Planning                       512  -0.0488    -0.2114     0.

In [77]:
ci_df.to_csv("nps_cause_select_bootstrap_ci.csv", index=False)
print("\nSaved: nps_cause_select_bootstrap_ci.csv")


Saved: nps_cause_select_bootstrap_ci.csv


In [78]:
ctrl_scenarios = [
    ("Equipment change",                 {"has_equipment_change":1},  {"has_equipment_change":0}),
    ("Fleet: RUB vs TGH (Continental)", {"equip_RUB":1,"equip_E320":0,"is_channel":0},
                                        {"equip_RUB":0,"equip_E320":0,"is_channel":0}),
    ("Fleet: E320 vs E300 (Channel)",   {"equip_E320":1,"equip_RUB":0,"is_channel":1},
                                        {"equip_E320":0,"equip_RUB":0,"is_channel":1}),
    ("Occupancy >400 pax",              {"total_pax_breach_400":1},  {"total_pax_breach_400":0}),
    ("Has connection",                  {"has_connection":1},         {"has_connection":0}),
    ("Channel vs Continental",          {"is_channel":1},             {"is_channel":0}),
    ("Price +£50 from mean",            {"metadata_price": X_raw["metadata_price"].mean()+50}, {}),
    ("Price +£100 from mean",           {"metadata_price": X_raw["metadata_price"].mean()+100}, {}),
]

ctrl_rows = []
for label, scen_vals, base_vals_override in ctrl_scenarios:
    base_overrides = {**zero_base_raw, **base_vals_override}
    scen_overrides = {**zero_base_raw, **base_vals_override, **scen_vals}
    base_pt, _, _   = nps_raw(base_overrides)
    scen_pt, lo, hi = nps_raw(scen_overrides)
    ctrl_rows.append({"variable": label, "baseline_nps": base_pt, "scenario_nps": scen_pt,
                      "nps_change": round(scen_pt - base_pt, 2),
                      "ci_lower": round(lo - base_pt, 2), "ci_upper": round(hi - base_pt, 2)})

ctrl_df = pd.DataFrame(ctrl_rows)
print("\n CONTROLLABLE VARIABLES ")
print(ctrl_df.sort_values("nps_change").to_string(index=False))


 CONTROLLABLE VARIABLES 
                       variable  baseline_nps  scenario_nps  nps_change  ci_lower  ci_upper
               Equipment change         46.80         37.07       -9.73    -11.57     -8.06
          Price +£100 from mean         46.33         37.29       -9.04     -9.93     -8.18
           Price +£50 from mean         46.33         41.90       -4.43     -5.18     -3.72
             Occupancy >400 pax         46.65         42.45       -4.20     -5.57     -2.85
  Fleet: E320 vs E300 (Channel)         53.43         50.86       -2.57     -3.99     -1.10
                 Has connection         46.56         44.37       -2.19     -3.34     -1.01
Fleet: RUB vs TGH (Continental)         34.97         46.70       11.73      8.87     14.47
         Channel vs Continental         33.83         52.47       18.64     17.27     20.05


In [79]:
# get baseline means for continuous vars (Channel only where relevant)
channel_mask = df["is_channel"] == 1

ctrl_scenarios_continuous = [

    ("Clean score: 90→95 (Channel)",
     {"clean_score_routine_channel": 95, "is_channel": 1},
     {"clean_score_routine_channel": 90, "is_channel": 1}),

    ("Clean score: 85→95 (Channel)",
     {"clean_score_routine_channel": 95, "is_channel": 1},
     {"clean_score_routine_channel": 85, "is_channel": 1}),

    ("Hours since last clean: +4hrs",
     {"hours_since_last_clean": df["hours_since_last_clean"].mean() + 4}, {}),

    ("PM interior: 30→60 days since",
     {"pm_days_since_interior": 60}, {"pm_days_since_interior": 30}),

    ("PM catering: 30→60 days since",
     {"pm_days_since_catering": 60}, {"pm_days_since_catering": 30}),

    ("PM comms: 30→60 days since",
     {"pm_days_since_comms": 60}, {"pm_days_since_comms": 30}),

    ("PM toilet: 30→60 days since",
     {"pm_days_since_toilet": 60}, {"pm_days_since_toilet": 30}),

    ("PM climate: 30→60 days since",
     {"pm_days_since_climate": 60}, {"pm_days_since_climate": 30}),

    # --- OPEN FAULTS (cm_open_*) ---
    # each represents: 1 additional open fault of that type vs 0
    ("Open fault: catering (+1)",
     {"cm_open_catering": df["cm_open_catering"].mean() + 1}, {}),

    ("Open fault: comms (+1)",
     {"cm_open_comms": df["cm_open_comms"].mean() + 1}, {}),

    ("Open fault: climate (+1)",
     {"cm_open_climate": df["cm_open_climate"].mean() + 1}, {}),

    ("Open fault: interior (+1)",
     {"cm_open_interior": df["cm_open_interior"].mean() + 1}, {}),

    ("Open fault: cleaning (+1)",
     {"cm_open_cleaning": df["cm_open_cleaning"].mean() + 1}, {}),
]


for label, scen_vals, base_vals_override in ctrl_scenarios_continuous:
    base_overrides = {**zero_base_raw, **base_vals_override}
    scen_overrides = {**zero_base_raw, **base_vals_override, **scen_vals}
    base_pt, _, _   = nps_raw(base_overrides)
    scen_pt, lo, hi = nps_raw(scen_overrides)
    ctrl_rows.append({
        "variable":     label,
        "baseline_nps": base_pt,
        "scenario_nps": scen_pt,
        "nps_change":   round(scen_pt - base_pt, 2),
        "ci_lower":     round(lo - base_pt, 2),
        "ci_upper":     round(hi - base_pt, 2),
    })

ctrl_df_full = pd.DataFrame(ctrl_rows)
print("\n CONTROLLABLE VARIABLES (full) ")
print(ctrl_df_full.sort_values("nps_change").to_string(index=False))
ctrl_df_full.to_csv("nps_controllable_vars.csv", index=False)


 CONTROLLABLE VARIABLES (full) 
                       variable  baseline_nps  scenario_nps  nps_change  ci_lower  ci_upper
               Equipment change         46.80         37.07       -9.73    -11.57     -8.06
          Price +£100 from mean         46.33         37.29       -9.04     -9.93     -8.18
           Price +£50 from mean         46.33         41.90       -4.43     -5.18     -3.72
             Occupancy >400 pax         46.65         42.45       -4.20     -5.57     -2.85
  Fleet: E320 vs E300 (Channel)         53.43         50.86       -2.57     -3.99     -1.10
                 Has connection         46.56         44.37       -2.19     -3.34     -1.01
       Open fault: climate (+1)         46.33         46.17       -0.16     -0.83      0.50
   PM climate: 30→60 days since         46.52         46.50       -0.02     -0.73      0.70
  Hours since last clean: +4hrs         46.33         46.31       -0.02     -0.69      0.61
     PM comms: 30→60 days since         46.44  

In [82]:
import statsmodels.formula.api as smf
from scipy import stats

X_brant    = result_raw.model.exog
y_full     = result_raw.model.endog
feat_names = core_vars_raw

y_b1 = (y_full >= 1).astype(int)
y_b2 = (y_full >= 2).astype(int)

def fit_binary(X, y, label=""):
    Xc = sm.add_constant(X, has_constant='add')
    res = sm.Logit(y, Xc).fit(disp=0, method='bfgs', maxiter=500)
    if not res.mle_retvals['converged']:
        print(f"  {label}: BFGS failed, retrying with Newton...")
        res = sm.Logit(y, Xc).fit(disp=0, method='newton', maxiter=500)
    return res

print("Fitting binary logit at cut-point 1 (Detractor vs Passive+Promoter)...")
m1 = fit_binary(X_brant, y_b1, "cut-point 1")
print("Fitting binary logit at cut-point 2 (Detractor+Passive vs Promoter)...")
m2 = fit_binary(X_brant, y_b2, "cut-point 2")

ol_coefs = result_raw.params[:len(feat_names)].values
b1_coefs = m1.params[1:]        # ndarray, skip intercept
b2_coefs = m2.params[1:]
b1_var   = np.diag(m1.cov_params())[1:]
b2_var   = np.diag(m2.cov_params())[1:]

chi2_vals = (b1_coefs - b2_coefs)**2 / (b1_var + b2_var)
p_vals    = 1 - stats.chi2.cdf(chi2_vals, df=1)

brant_df = pd.DataFrame({
    "variable":  feat_names,
    "ol_coef":   ol_coefs,
    "b1_coef":   b1_coefs,
    "b2_coef":   b2_coefs,
    "chi2":      chi2_vals,
    "p_value":   p_vals,
    "violation": p_vals < 0.05
})
brant_df = brant_df.sort_values("p_value")

print("\n BRANT TEST RESULTS ")
print(f"{'Variable':<35} {'OL coef':>8} {'B1 coef':>8} {'B2 coef':>8} {'chi2':>7} {'p':>7} {'Violation':>10}")
for _, r in brant_df.iterrows():
    flag = " FAIL" if r.violation else ""
    print(f"{r.variable:<35} {r.ol_coef:>8.4f} {r.b1_coef:>8.4f} {r.b2_coef:>8.4f} "
          f"{r.chi2:>7.3f} {r.p_value:>7.4f}{flag}")

n_fail = brant_df.violation.sum()
print(f"\nVariables failing PO assumption (p<0.05): {n_fail} / {len(feat_names)}")
print("Global impression: proportional odds",
      "LIKELY VIOLATED" if n_fail > len(feat_names)*0.2 else "broadly defensible")

brant_df.to_csv("brant_test_results.csv", index=False)
print("Saved: brant_test_results.csv")

Fitting binary logit at cut-point 1 (Detractor vs Passive+Promoter)...


C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


Fitting binary logit at cut-point 2 (Detractor+Passive vs Promoter)...


C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
C:\Users\gehan\anaconda3\envs\eurostar\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  cut-point 2: BFGS failed, retrying with Newton...

 BRANT TEST RESULTS 
Variable                             OL coef  B1 coef  B2 coef    chi2       p  Violation
metadata_price                       -0.0025  -0.0036  -0.0021 132.289  0.0000 FAIL
equip_RUB                             0.3240   0.1707   0.3764  18.695  0.0000 FAIL
has_equipment_change                 -0.2710  -0.3951  -0.2253  18.103  0.0000 FAIL
pm_days_since_catering                0.0005   0.0010   0.0004  10.238  0.0014 FAIL
pm_days_since_comms                  -0.0000  -0.0002   0.0000   8.874  0.0029 FAIL
cause_Eurostar_Operations            -0.1322  -0.2375  -0.1004   7.803  0.0052 FAIL
has_connection                       -0.0628  -0.0052  -0.0823   7.430  0.0064 FAIL
cause_Weather_Safety                 -0.0368  -0.1425  -0.0000   5.706  0.0169 FAIL
clean_score_deep                     -0.0005  -0.0011  -0.0003   5.572  0.0183 FAIL
cause_Rolling_Stock                  -0.2598  -0.3273  -0.2379   4.416  0.0356 F